# Week 11 lecture demo: same data, different prompts
**No live prompting.** Every AI response below was generated ahead of time (Claude, fresh chat, no tools, Nov 2 2026) and saved in this folder as `prompt_*.txt` / `response_*.md`. The cells *run the code the AI wrote* against the real file and compare with ground truth.

Jump to: [Ground truth](#1.-Ground-truth) · [Prompt B (specific)](#2.-Specific-prompt-(B)) · [Why not a plain median?](#3.-Why-not-a-plain-median?) · [Prompt C (one big ask)](#4.-One-big-ask-(C)) · [Prompt D (plan first)](#5.-Plan-first-(D))

## 1. Ground truth
Computed by hand (not by AI): Kaplan-Meier median overall survival by collapsed stage, TCGA LUAD, 566 patients.

In [1]:
import pandas as pd, numpy as np
from lifelines import KaplanMeierFitter
pd.set_option('display.width', 160)

df = pd.read_csv("data_clinical_patient.txt", sep="\t", comment="#")
d = df.dropna(subset=["OS_MONTHS", "OS_STATUS"]).copy()
d["event"] = d.OS_STATUS.str.startswith("1").astype(int)
d["stage"] = d.AJCC_PATHOLOGIC_TUMOR_STAGE.str.extract(r"STAGE (IV|III|II|I)")[0]
d = d.dropna(subset=["stage"])
truth = []
for s in ["I", "II", "III", "IV"]:
    g = d[d.stage == s]
    k = KaplanMeierFitter().fit(g.OS_MONTHS, g.event)
    truth.append((s, len(g), int(g.event.sum()), round(k.median_survival_time_, 1)))
truth = pd.DataFrame(truth, columns=["stage", "n", "events", "median_os_months"])
print(f"{len(df)} patients in file, {len(d)} usable (stage, OS time and status all present)")
truth

566 patients in file, 503 usable (stage, OS time and status all present)


,stage,n,events,median_os_months
0,I,274,67,86.1
1,II,122,53,40.6
2,III,80,46,26.5
3,IV,27,16,27.2


## 2. Specific prompt (B)
Prompt: `prompt_B.txt` (file format, columns, event coding, KM not plain median, how to handle missing, exact output). Below is the code it returned, run as-is.

In [2]:
%run B_code.py

Dropped 63 of 566 patients; 503 remain.

stage   n  events  median_os_months
    I 274      67              86.1
   II 122      53              40.6
  III  80      46              26.5
   IV  27      16              27.2


In [3]:
# Does the AI's table match ground truth?
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    %run B_code.py
b = pd.read_fwf(io.StringIO(buf.getvalue().split("\n\n",1)[1]))
print("Matches ground truth:", (b.values == truth.values).all())

Matches ground truth: True


## 3. Why not a plain median?
The specific prompt said *use Kaplan-Meier because patients are censored*. Here is what the obvious shortcut gives instead.

In [4]:
rows = []
for st in ["I", "II", "III", "IV"]:
    g = d[d.stage == st]
    rows.append({"stage": st,
                 "median_all_ignoring_censoring": round(g.OS_MONTHS.median(), 1),
                 "median_deceased_only": round(g[g.event == 1].OS_MONTHS.median(), 1)})
naive = pd.DataFrame(rows).set_index("stage")
naive.join(truth.set_index("stage")[["n", "median_os_months"]]).rename(columns={"median_os_months": "KM (correct)"})

,median_all_ignoring_censoring,median_deceased_only,n,KM (correct)
stage,,,,
I,22.9,28.1,274,86.1
II,22.3,16.4,122,40.6
III,15.6,13.1,80,26.5
IV,20.0,20.8,27,27.2


Stage I is wrong by a factor of 4, and the ordering is wrong too. Also note Stage IV (n=27) lands near Stage III in the correct table: tiny groups give noisy medians.

## 4. One big ask (C)
Prompt: `prompt_C.txt`: *"Clean this dataset, run a survival analysis by stage, make a plot, and write a methods paragraph."*  Read `response_C.md` first, then run its code. Only two edits: the file name and a headless plotting backend.

In [5]:
%run C_code.py

n start 566 after NA drop 503 after time>0 499
I 272 67 86.13604235
II 122 53 40.60229477
III 79 45 28.89831344
IV 26 16 27.15586679
p 3.0125426172485134e-12


**What to look for**
- It ran and produced a plausible plot. Nothing crashed.
- It dropped **4 more patients** than the specific prompt did (`time > 0`). Nobody asked for that. Stage III is now n=79 with median 28.9, not n=80 and 26.5. The code never reports the count.
- Its methods paragraph says a **Cox model** was fit and **Schoenfeld residuals** checked. The code does neither.

## 5. Plan first (D)
Prompt: `prompt_D.txt`: *plan and checks first, code for step 1 only*. Read the plan in `response_D.md`. Run step 1:

In [6]:
%run D_code.py

(566, 38)
['PATIENT_ID', 'SUBTYPE', 'CANCER_TYPE_ACRONYM', 'OTHER_PATIENT_ID', 'AGE', 'SEX', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'AJCC_STAGING_EDITION']
Unique patients: 566 of 566
AJCC_PATHOLOGIC_TUMOR_STAGE    54
OS_STATUS                      52
OS_MONTHS                      61
dtype: int64


In [7]:
# One of its own checks: "row count matches wc -l minus 5"
import subprocess
n = int(subprocess.check_output(["wc", "-l", "data_clinical_patient.txt"]).split()[0])
print("wc -l minus 5 =", n - 5, "| rows pandas read =", len(pd.read_csv("data_clinical_patient.txt", sep="\t", skiprows=4)))

wc -l minus 5 = 565 | rows pandas read = 566


Off by one: the file has no trailing newline, so `wc -l` undercounts. The AI's check was sensible and still needed checking.